# Preclass
AI for Materials Science — Hands-on session 3

Read a PDF and identify a material–property relationship.

**Result:** `sections.json` and a hand-drawn graph.

Run top to bottom. No API key needed.

In [ ]:
%pip -q install pymupdf pandas
import json, pymupdf, pandas as pd
from pathlib import Path
from google.colab import files
from IPython.display import Image, display

## 2. Upload a PDF
Run the cell → click **Choose Files** below it → select one PDF.

In [ ]:
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Choose one PDF for this introductory notebook.')
name, data = next(iter(uploaded.items()))
PDF = Path(Path(name).name)
PDF.write_bytes(data)
doc = pymupdf.open(PDF)
if doc.needs_pass:
    raise ValueError('Upload an unlocked PDF.')
print(PDF.name, '|', len(doc), 'pages')

## 3. Read pages
`None`: all pages. `[3, 4, 5]`: selected PDF viewer pages (starting at 1). Scanned pages need OCR.

In [ ]:
PAGES = None
selected = list(range(1, len(doc) + 1)) if PAGES is None else PAGES
if any(p < 1 or p > len(doc) for p in selected):
    raise ValueError(f'Choose pages within 1–{len(doc)}.')
chunks = [{'paper_id': PDF.stem, 'page': p, 'text': '\n\n'.join(b[4].strip() for b in doc[p - 1].get_text('blocks', sort=False) if b[6] == 0)} for p in selected]
display(pd.DataFrame([{'page': c['page'], 'characters': len(c['text'])} for c in chunks]))
print('Pages without text (OCR may be needed):', [c['page'] for c in chunks if not c['text']])

## 4. Check the original page
Change `PAGE`; compare reading order, numbers and units.

In [ ]:
PAGE = 1
page = doc[PAGE - 1]
display(Image(page.get_pixmap(matrix=pymupdf.Matrix(1.2, 1.2)).tobytes('png')))
print('\n\n'.join(b[4].strip() for b in page.get_text('blocks', sort=False) if b[6] == 0)[:3000])

## 5. Save the parsed text
Download the text with its PDF page numbers.

In [ ]:
Path('sections.json').write_text(json.dumps(chunks, ensure_ascii=False, indent=2), encoding='utf-8')
files.download('sections.json')

## Pre-lab submission
Submit five paper titles and a hand-drawn graph with values, units and conditions. Bring the PDFs to class.